AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 03. CNN 모델에 데이터 증강 적용하기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 4강 CNN 구조를 활용해 회전 증강이 포함된 새 모델을 만들 수 있다.
- 증강 레이어와 정규화 레이어의 배치 순서를 설명할 수 있다.
- 훈련과 검증·예측에서 데이터 증강의 동작이 다른 이유를 이해할 수 있다.

### VS Code 실행 준비

4강 저장 모델의 구조를 사용해 회전 증강이 포함된 CNN을 구성합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [1]:
from tensorflow import keras
from tensorflow.keras import layers
IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]

In [2]:
# 회전 증강 레이어 생성
data_augmentation = keras.Sequential([
    layers.RandomRotation(
        factor=0.25,
        fill_mode="constant",
        fill_value=0.0,
        interpolation="nearest"
    )
])

## 1. 기존 CNN 모델 살펴보기

4강에서는 웨이퍼맵 이미지를 읽고 CNN 모델을 만들어 학습했다. 이번 미션에서는 **기존 모델을 새로 설계하지 않고, 앞에 회전 증강 레이어 하나만 추가**한다.

4강 노트북에서 모델을 정의한 코드를 찾아 아래 순서를 확인해보자.

입력 이미지 (64 × 64 × 3)

↓

Rescaling(1./255) 픽셀값 정규화

↓

Conv2D → MaxPooling2D 특징 추출

↓

Conv2D → MaxPooling2D 특징 추출

↓

Flatten → Dense → Dense 클래스 예측

4강에서 사용한 `train_ds`, `val_ds`, `test_ds`, `IMG_SIZE`, `EPOCHS`, `class_names`를 그대로 활용한다. 런타임을 다시 시작했다면 데이터셋 생성 셀부터 실행한다.

## 2. 증강 레이어는 어디에 넣어야 할까?

Mission 01에서 만든 `data_augmentation`을 모델의 입력 부분에 추가하자. 이미지의 픽셀값을 정규화하기 **전**에 증강하도록 배치한다.

입력 이미지 원본 픽셀값 범위 0~255

↓

data_augmentation RandomRotation · 새로 추가

↓

Rescaling(1./255) 정규화

↓

기존 CNN 나머지 레이어 구조는 변경하지 않음

`fill_value=0.0`은 회전 후 빈 공간을 검은색으로 채우기 위한 설정이다. 현재처럼 증강을 정규화 전에 수행하면 원본 이미지의 픽셀값 범위에 맞추어 설정을 이해할 수 있다.

증강을 반드시 정규화 앞에 넣어야 하는 것은 아니다. 다만 레이어 순서에 따라 `fill_value` 등 픽셀값에 관한 설정을 맞추어야 한다. 이번 실습에서는 원래 사용했던 설정을 유지하기 위해 증강 → 정규화 순서로 작성한다.

## 3. 4강 코드를 수정해 새 모델 만들기
기존 `model`은 비교를 위해 그대로 두고, 증강 모델은 `model_aug`라는 이름으로 만들자. 아래 코드에는 **추가할 위치와 기존 구조를 유지할 부분**을 표시했다.

In [3]:
# Mission 01에서 data_augmentation을 이미 생성했다고 가정
model_aug = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    data_augmentation,  # 학습 시에만 회전 변형 적용           # 추가: 회전 증강
    layers.Rescaling(1./255),      # 기존: 정규화

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(class_names), activation="softmax"),
])

**코드 작성 미션**

1. 4강에서 사용한 CNN 모델 정의 코드를 찾는다.
2. 기존 모델을 지우지 않고 `model_aug`를 새로 만든다.
3. 입력 레이어 다음에 `data_augmentation`을 추가한다.
4. 정규화와 나머지 CNN 레이어를 4강과 동일한 구조로 작성한다.
5. `model_aug.summary()`로 구조를 확인한다.

In [4]:
model_aug.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 822,597 (3.14 MB)

 Trainable params: 822,597 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

Mission 01에서는 `data_augmentation`을 `keras.Sequential`로 묶었으므로, `summary()`에는 내부의 `RandomRotation`이 아니라 **중첩된 Sequential**이 하나의 레이어처럼 표시될 수 있다.

**증강 레이어를 묶은 경우**

model_aug (Sequential)

data_augmentation (Sequential)

RandomRotation

Rescaling → CNN

`Sequential` 안에 또 하나의 `Sequential`이 들어갈 수 있다.

**증강 레이어를 직접 넣은 경우**

model_aug (Sequential)

RandomRotation

Rescaling → CNN

증강이 하나뿐이라면 모델에 `RandomRotation`을 직접 추가해도 된다.

## 4. 증강은 언제 적용될까?

모델 안에 증강 레이어를 넣더라도 모든 실행에서 무작위 회전이 발생하는 것은 아니다. **훈련할 때만 활성화되고, 검증·평가·예측할 때는 비활성화**된다.

| 실행 | 모드 | RandomRotation 동작 |
| --- | --- | --- |
| `model_aug.fit(train_ds, ...)`의 훈련 단계 | 훈련 모드 | 회전 적용 |
| `fit()` 내부의 검증 단계 | 추론 모드 | 회전 미적용 |
| `model_aug.evaluate(test_ds)` | 추론 모드 | 회전 미적용 |
| `model_aug.predict(test_ds)` | 추론 모드 | 회전 미적용 |

`training=True`를 직접 사용했던 Mission 02와 달리, `fit()`은 훈련 단계와 검증 단계를 상황에 맞는 모드로 처리한다. 따라서 **검증용·테스트용 이미지에 별도의 회전 증강을 적용하지 않는다.**

이번 미션에서는 모델 구조를 만드는 것까지만 수행한다. 학습과 성능 비교는 다음 실습 단계에서 진행한다.

## 5. 코드 이해 활동

**Q3.1.**

4강 CNN 코드에서 회전 증강 레이어를 추가한 위치를 찾으시오. `Rescaling` 앞에 배치한 이유를 픽셀값과 연결해 설명하시오.

**답변**

`data_augmentation`은 **입력층 다음, `Rescaling` 앞**에 있습니다. 원본 픽셀 범위인 0~255에서 회전하고 빈 곳을 0으로 채운 뒤, 픽셀값을 0~1로 정규화합니다.

**Q3.2.**

4강 모델과 새로 만든 `model_aug`의 `summary()`를 비교하시오. 새 모델에 추가된 항목은 무엇이며, 기존 CNN 레이어의 구성은 동일한가?

**답변**

새 모델에는 회전 증강을 묶은 `Sequential`이 추가됐습니다. 정규화 이후의 `Conv2D`·`MaxPooling2D`·`Flatten`·`Dense` 구성은 4강 CNN과 같습니다.

**Q3.3.**

`data_augmentation`을 모델에 넣었을 때 `summary()`에서 `Sequential`로 표시되는 이유를 설명하시오. 이 안에는 어떤 레이어가 들어 있는가?

**답변**

`data_augmentation`을 하나의 `Sequential`로 묶어 모델 안에 넣었기 때문입니다. 그 안에는 `RandomRotation` 한 개가 들어 있습니다.

**Q3.4.**

`fit()`에서 훈련 데이터와 검증 데이터가 모델에 입력될 때, `RandomRotation`의 동작은 어떻게 다른가? 이 차이가 필요한 이유를 설명하시오.

**답변**

훈련에서는 무작위 회전을 적용해 다양한 방향을 학습합니다. 검증에서는 회전을 적용하지 않아, 원본 이미지 기준으로 성능을 비교할 수 있습니다.

**Q3.5.**

기존 4강 모델을 그대로 수정하지 않고 `model_aug`를 별도로 만드는 이유는 무엇인가? 이후 두 모델을 비교할 때 동일하게 유지해야 할 조건을 두 가지 작성하시오.

**답변**

기존 모델을 기준으로 보존하고, 증강 CNN을 새 가중치에서 학습해 비교하기 위해서입니다. 두 모델의 **데이터 분할과 CNN 구조**, 학습 설정을 동일하게 유지합니다.